# The Converse API

`converse` is the modern, unified way to call any Bedrock model. Instead of hand-building a different JSON payload for each model family (like we did with `invoke_model`), you use one consistent structure: `messages`, `system`, and `inferenceConfig`. Bedrock translates it to whatever the underlying model expects.

This notebook covers:
1. `converse` - a single request/response call
2. The response structure
3. `converse_stream` - the same call, streamed

## 1. converse

Note the standardized shape:
- `messages` - the conversation turns
- `system` - system instructions that steer the model
- `inferenceConfig` - standardized parameters (temperature, maxTokens) that work across models
- `additionalModelRequestFields` - model-specific parameters (here, Claude's `top_k`)

> **Note:** the slide also passes `topP: 0.9`, but Claude Sonnet 4.5 rejects `temperature` and `topP` together (`ValidationException: ... cannot both be specified`). We use `temperature` only. This shows that while `inferenceConfig` is standardized, individual models still enforce their own rules.

In [ ]:
import boto3
import json

bedrock_client = boto3.client("bedrock-runtime", region_name="us-east-1")

response = bedrock_client.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{
        "role": "user",
        "content": [{"text": "Create a script to resize images."}]
    }],
    system=[{"text": "You are an app developer proficient in Python. "
                     "Only engage in discussion of coding topics."}],
    # Slide also shows "topP": 0.9 - but Sonnet 4.5 rejects it with temperature.
    inferenceConfig={"temperature": 0.7, "maxTokens": 500},  # "topP": 0.9,
    additionalModelRequestFields={"top_k": 200}
)

print(response["output"]["message"]["content"][0]["text"])

## 2. The response structure

The Converse response has a standardized shape too: `output` (the message), `stopReason`, `usage` (token counts), and `metrics` (latency). Below we show it with the generated text replaced by a placeholder and `ResponseMetadata` collapsed (matches the slide).

In [ ]:
import copy

view = copy.deepcopy(response)
view["ResponseMetadata"] = "{...}"
view["output"]["message"]["content"][0]["text"] = "<model-output>"

print(json.dumps(view, indent=4, default=str))

## 3. converse_stream

Same request, streamed. The response is an iterator of typed events. The text arrives in `contentBlockDelta` events; the rest (`messageStart`, `contentBlockStop`, `messageStop`, `metadata`) are structural.

In [ ]:
response = bedrock_client.converse_stream(
    modelId="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{
        "role": "user",
        "content": [{"text": "Create a script to resize images."}]
    }],
    system=[{"text": "You are an app developer proficient in Python. "
                     "Only engage in discussion of coding topics."}],
    inferenceConfig={"temperature": 0.7, "maxTokens": 500},
    additionalModelRequestFields={"top_k": 200}
)

for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)

The event types that make up the stream, in order:

In [ ]:
response = bedrock_client.converse_stream(
    modelId="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{
        "role": "user",
        "content": [{"text": "Create a script to resize images."}]
    }],
    system=[{"text": "You are an app developer proficient in Python. "
                     "Only engage in discussion of coding topics."}],
    inferenceConfig={"temperature": 0.7, "maxTokens": 500},
    additionalModelRequestFields={"top_k": 200}
)

for event in response["stream"]:
    print(list(event.keys())[0])